# Importing Libraries

In [1]:
## Import modules
import os, sys
import numpy as np
import geopandas as gpd
import cftime
import gc
import shapely
import json
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as pc
from ipywidgets import interact, IntSlider, Dropdown, VBox, HBox
import ipywidgets as widgets

# Add the directory containing 'cmct' to the Python path
# Navigate two levels up to reach main CmCt dir
cmct_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))

# Initialising Logger
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Import utilities for this comparison
sys.path.insert(0, cmct_dir)
from cmct.time_utils import *
from cmct.ice_area_extent import *
from cmct.ice_area_extent_modules.interpolation import *
from cmct.ice_area_extent_modules.residual_calculation import *
# from cmct.ice_area_extent_modules.json_to_netcdf import *
# from cmct.shapefile_utils import *

# Force initial garbage collection
gc.collect()

50

In [2]:
# Reload modules to pick up any changes to imports
import importlib
import cmct.ice_area_extent
import cmct.ice_area_extent_modules.residual_calculation
import cmct.ice_area_extent_modules.plotting_utils
from cmct.ice_area_extent_modules.plotting_utils import *
from cmct.ice_area_extent import calculate_basin_statistics, format_basin_stats
from cmct.ice_area_extent import calculate_basin_statistics
importlib.reload(cmct.ice_area_extent)
importlib.reload(cmct.ice_area_extent_modules.residual_calculation)
importlib.reload(cmct.ice_area_extent_modules.plotting_utils)

# Re-import to ensure functions are available
from cmct.ice_area_extent import *

# Import the new zoom-preserving functions
from cmct.ice_area_extent_modules.plotting_utils import (
    create_zoom_preserving_residual_widget,
    create_example_zoom_preserving_dashboard,
    create_interactive_residual_plot,  # Updated with zoom preservation
    create_basin_statistics_plot,
    create_time_series_plot,
    create_relative_time_series_plot,
    create_observations_model_residual_grid,
    create_correlation_matrix,
)

# CONFIGURATION

In [3]:
# Observation Dataset
# Ice sheet
loc = "GIS"  # 'GIS' or 'AIS'

# Set the observation data dir path
obs_filename = cmct_dir + "/data/ice_area_extent/observed_icemask_ismip_annual.nc"

# To use aggregation functions for basin
basin_aggregation = True  # IMPORTANT

basin_filename = cmct_dir + "/bin/ice_area_extent/GRE_Basins_IMBIE2_v1.3/GRE_Basins_IMBIE2_v1.3.shp"

# Set the Model Data dir path
# model_filename = cmct_dir + "/test/ice_area_extent/ensemble/sftgif_B001_hist.nc"
model_filename = cmct_dir + "/test/ice_area_extent/sftgif_GIS_JPL_ISSM_historical.nc"

# Set time range for comparison
start_year = 2006
end_year = 2015

# List of basins (ex ["NW", "NE"]) to compare if all -> "all", if none -> False
# If you do not know which basins are in the model, you can put "auto"
# NOTE: Align this list with the basins in the model.
basin_list = "all"

# Output filetype and filename
filetype = "netcdf"  # netcdf or json or None
filename = "ice_area_extent_comparison"

# Optional Configurations
interpolation_method = "slinear"  # 'nearest', 'linear', 'cubic'
accuracy_calculation_method = "mean"  # 'mean', 'RMS',

colors = {
    "CW": "blue",
    "NE": "red",
    "SE": "green",
    "SW": "orange",
    "NO": "purple",
    "NW": "brown",
}


# Loading all data files

In [4]:
# Check if observation file exist
if not os.path.exists(obs_filename):
    raise FileNotFoundError(f"Observation file not found: {obs_filename}")

# # Check if model file exist
if not os.path.exists(model_filename):
    raise FileNotFoundError(f"Model file not found: {model_filename}")



if basin_aggregation and not os.path.exists(basin_filename):
    raise FileNotFoundError(f"Basin shapefile not found: {basin_filename}")
    # Load basin shapes

print(basin_filename)
basins, basin_list = load_basins(basin_filename, basin_list)

print(obs_filename)
observations = load_observations_ice_area_extent(obs_filename, basins)

print(model_filename)
model_res = load_model_ice_area_extent(model_filename)

/Users/aditya_pachpande/Documents/GitHub/CmCt/bin/ice_area_extent/GRE_Basins_IMBIE2_v1.3/GRE_Basins_IMBIE2_v1.3.shp
/Users/aditya_pachpande/Documents/GitHub/CmCt/data/ice_area_extent/observed_icemask_ismip_annual.nc
/Users/aditya_pachpande/Documents/GitHub/CmCt/test/ice_area_extent/sftgif_GIS_JPL_ISSM_historical.nc


## Handelling Time Consistency

In [5]:
# Simplifying date data type
observations.ds["time"] = standardising_time_var(observations.time)
model_res.ds["time"] = standardising_time_var(model_res.time)

# Handelling Time Range
checking_ice_area_extent_daterange(observations.time.values, model_res.time.values, start_year, end_year)


The selected dates 2006 to 2015 are within the overlapping data range.


# Interpolation

In [6]:
interpolater = Interpolater(model_res, observations)
model_res.ds = interpolater.interpolate()

2025-08-06 12:08:25,214 - INFO - Input x coordinates: [-720000. -715000. -710000. -705000. -700000. -695000. -690000. -685000.
 -680000. -675000. -670000. -665000. -660000. -655000. -650000. -645000.
 -640000. -635000. -630000. -625000. -620000. -615000. -610000. -605000.
 -600000. -595000. -590000. -585000. -580000. -575000. -570000. -565000.
 -560000. -555000. -550000. -545000. -540000. -535000. -530000. -525000.
 -520000. -515000. -510000. -505000. -500000. -495000. -490000. -485000.
 -480000. -475000. -470000. -465000. -460000. -455000. -450000. -445000.
 -440000. -435000. -430000. -425000. -420000. -415000. -410000. -405000.
 -400000. -395000. -390000. -385000. -380000. -375000. -370000. -365000.
 -360000. -355000. -350000. -345000. -340000. -335000. -330000. -325000.
 -320000. -315000. -310000. -305000. -300000. -295000. -290000. -285000.
 -280000. -275000. -270000. -265000. -260000. -255000. -250000. -245000.
 -240000. -235000. -230000. -225000. -220000. -215000. -210000. -20500

# Comparison and Residual Calculation 

In [7]:
print(type(basins))
years = np.arange(start_year, end_year + 1)

residuals = create_ice_area_extent_dataset(observations, model_res, years, basins)

2025-08-06 12:08:29,604 - INFO - Computing basin mask once for all ensemble members...
2025-08-06 12:08:29,611 - INFO - Transformed basin CW: 1211 points
2025-08-06 12:08:29,624 - INFO - Transformed basin NE: 6780 points
2025-08-06 12:08:29,636 - INFO - Transformed basin SE: 15619 points
2025-08-06 12:08:29,643 - INFO - Transformed basin SW: 4063 points
2025-08-06 12:08:29,666 - INFO - Transformed basin NO: 4016 points
2025-08-06 12:08:29,675 - INFO - Transformed basin NW: 4986 points
2025-08-06 12:08:29,684 - INFO - Grid dimensions: 2880 x 1680
2025-08-06 12:08:29,684 - INFO - Using year 2006 for basin mask creation
2025-08-06 12:08:29,684 - INFO - Creating basin mask...


<class 'dict'>


2025-08-06 12:09:45,997 - INFO - Basin assignment complete. Unique basin IDs: [-1  0  1  2  3  4  5]
2025-08-06 12:09:46,000 - INFO -   Unassigned points: 3112422
2025-08-06 12:09:46,002 - INFO -   Basin 0 (CW): 232302 points
2025-08-06 12:09:46,004 - INFO -   Basin 1 (NE): 478030 points
2025-08-06 12:09:46,006 - INFO -   Basin 2 (SE): 294627 points
2025-08-06 12:09:46,007 - INFO -   Basin 3 (SW): 217562 points
2025-08-06 12:09:46,009 - INFO -   Basin 4 (NO): 232587 points
2025-08-06 12:09:46,010 - INFO -   Basin 5 (NW): 270870 points
2025-08-06 12:09:46,012 - INFO - Basin assignment rate: 1725978/4838400 (35.7%)
2025-08-06 12:09:46,012 - INFO - Creating ice_area_extent dataset with precomputed basin mask...
2025-08-06 12:09:46,981 - INFO - observations data shape: (10, 2880, 1680)
2025-08-06 12:09:49,426 - INFO - Model data shape after alignment: (10, 2880, 1680)
2025-08-06 12:09:49,427 - INFO - Basin mask shape: (2880, 1680)
2025-08-06 12:09:49,427 - INFO - Computing residuals...
202

In [8]:
residuals = load_residuals(residuals)


In [9]:
vars(residuals)


{'ds': <xarray.Dataset> Size: 774MB
 Dimensions:                 (time: 10, y: 2880, x: 1680, basin_id: 6)
 Coordinates:
   * time                    (time) int64 80B 2006 2007 2008 ... 2013 2014 2015
   * x                       (x) float32 7kB -7.195e+05 -7.185e+05 ... 9.595e+05
   * y                       (y) float32 12kB -3.45e+06 -3.448e+06 ... -5.705e+05
     basin_names             (basin_id) <U2 48B 'CW' 'NE' 'SE' 'SW' 'NO' 'NW'
 Dimensions without coordinates: basin_id
 Data variables:
     residual                (time, y, x) float32 194MB 0.0 0.0 0.0 ... 0.0 0.0
     basin                   (time, y, x) int32 194MB -1 -1 -1 -1 ... -1 -1 -1 -1
     observations_ice_mask   (time, y, x) float32 194MB 0.0 0.0 0.0 ... 0.0 0.0
     model_ice_mask          (time, y, x) float32 194MB 0.0 0.0 0.0 ... 0.0 0.0
     stats_avg_abs_residual  (time) float64 80B 0.02461 0.02462 ... 0.02482
     stats_rms_residual      (time) float64 80B 0.1319 0.1319 ... 0.1326 0.1327
     stats_sum_residu

# Statistics Calculation

In [10]:
basin_stats = calculate_basin_statistics(residuals)
print(format_basin_stats(basin_stats))

2025-08-06 12:09:49,944 - INFO - Starting basin statistics calculation
2025-08-06 12:09:49,945 - INFO - Processing 10 time steps and 6 basins
2025-08-06 12:09:50,587 - INFO - Basin statistics calculation completed


=== Statistics for Year 2006 ===
Basin | Sum       | Count    | Mean        | Winsorized  | Outlier Wgt | Std        | RMS
-------------------------------------------------------------------------------------
CW    | 23.257492065429688 |   232302 |  0.00010012 |  0.00013375 |  0.00004944 |   0.036565 |   0.036565
NE    | -2797.58154296875 |   478030 | -0.00585231 | -0.00666378 | -0.00003793 |   0.115760 |   0.115908
SE    | 14.06719970703125 |   294627 |  0.00004775 | -0.00724717 | -0.00024722 |   0.204647 |   0.204647
SW    | 1017.2067260742188 |   217562 |  0.00467548 |  0.00493314 |  0.00028247 |   0.081600 |   0.081734
NO    | 2110.669189453125 |   232587 |  0.00907475 |  0.00970253 |  0.00030426 |   0.115014 |   0.115371
NW    | -181.97161865234375 |   270870 | -0.00067180 | -0.00062181 |  0.00009068 |   0.060439 |   0.060443
-------------------------------------------------------------------------------------


=== Statistics for Year 2007 ===
Basin | Sum       | Count    | Mean 

# Plot Generation

## Configuration
- residuals are required for plot generation

In [11]:
# Import plotting functions from plotting_utils
# (Already imported above, but keeping for reference)
from cmct.ice_area_extent_modules.plotting_utils import (
    create_time_series_plot,
    create_relative_time_series_plot,
    create_observations_model_residual_grid,
    create_correlation_matrix,
    create_zoom_preserving_residual_widget,  # New zoom-preserving widget
    create_example_zoom_preserving_dashboard,  # Helper function
)

# Configuration for plots
year = 2007  # Initial year for demonstration
cmap = "ocean"  # Leave default to Ocean :)
aspect = "auto"
basin_id = 0  # Initial basin for demonstration

# Plot config
plt.figure(figsize=(10, 6))

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

In [12]:
# Create initial plots to show functionality
create_interactive_residual_plot(residuals, year, basin_id=basin_id)
create_basin_statistics_plot(basin_stats, year)

# Create the zoom-preserving widget
zoom_preserving_widget = create_zoom_preserving_residual_widget(
    residuals, 
    basin_stats, 
    available_years=list(range(start_year, end_year + 1))
)

# Display the widget
display(zoom_preserving_widget)

In [13]:
statistic_dropdown = Dropdown(
    options=[('Mean', 'mean'), ('Standard Deviation', 'std'), ('RMS', 'rms'), 
             ('Winsorized Mean', 'winsorized_mean'), ('Outlier Weighted Mean', 'outlier_weighted_mean'), ('Sum', 'sum')],
    value='mean',
    description="Statistic:"
)

def interactive_grid_plot(statistic):
    """Interactive grid plotting function"""
    fig = create_observations_model_residual_grid(basin_stats, statistic, colors=colors)
    if fig:
        fig.show()

interact(interactive_grid_plot, statistic=statistic_dropdown)

# Add units to the plots

interactive(children=(Dropdown(description='Statistic:', options=(('Mean', 'mean'), ('Standard Deviation', 'st…

<function __main__.interactive_grid_plot(statistic)>

In [14]:
statistic_dropdown_rel = Dropdown(
    options=[('Mean', 'mean'), ('Standard Deviation', 'std'), ('RMS', 'rms'), 
             ('Winsorized Mean', 'winsorized_mean'), ('Outlier Weighted Mean', 'outlier_weighted_mean'), ('Sum', 'sum')],
    value='mean',
    description="Statistic:"
)

def interactive_relative_time_series(statistic):
    """Interactive relative time series plotting"""
    fig = create_relative_time_series_plot(basin_stats, statistic, colors=colors)
    if fig:
        fig.show()

interact(interactive_relative_time_series, statistic=statistic_dropdown_rel)

# Add another plot with all basins in one

# To consider a place of high outliers o rimp it would have a change based on 3x the normal rate

interactive(children=(Dropdown(description='Statistic:', options=(('Mean', 'mean'), ('Standard Deviation', 'st…

<function __main__.interactive_relative_time_series(statistic)>

In [ ]:
def create_box_whiskers_plot(residuals, statistic='residual', colors=None):
    """
    Create a box and whiskers plot aggregating across basins for each time period.
    
    Parameters:
    -----------
    residuals : Residual object
        Contains the residual data with basin_stats
    statistic : str
        The statistic to plot ('residual', 'mean', 'std', etc.)
    colors : dict, optional
        Color mapping for basins
        
    Returns:
    --------
    plotly figure
    """
    import plotly.graph_objects as go
    import plotly.express as px
    import numpy as np
    import pandas as pd
    from plotly.subplots import make_subplots
    
    # Get the basin statistics
    basin_stats_data = residuals.basin_stats if hasattr(residuals, 'basin_stats') else basin_stats
    
    # Prepare data for box plot
    plot_data = []
    
    # Get all years and basins
    years = sorted(basin_stats_data.keys())
    all_basins = set()
    for year_data in basin_stats_data.values():
        all_basins.update(year_data.keys())
    
    # Create data structure for box plot
    for year in years:
        year_data = basin_stats_data[year]
        values = []
        basin_names = []
        
        for basin_name, basin_data in year_data.items():
            if statistic in basin_data:
                values.append(basin_data[statistic])
                basin_names.append(basin_name)
        
        if values:  # Only add if we have data
            plot_data.append({
                'year': year,
                'values': values,
                'basin_names': basin_names
            })
    
    # Create the box plot
    fig = go.Figure()
    
    # Add box plots for each year (without individual points first)
    for data in plot_data:
        fig.add_trace(go.Box(
            y=data['values'],
            x=[str(data['year'])] * len(data['values']),
            name=str(data['year']),
            boxpoints=False,  # Don't show points on box plot
            fillcolor='lightblue',
            line=dict(color='darkblue'),
            opacity=0.7,
            showlegend=False
        ))
    
    # Now add individual colored points as scatter plots
    if colors:
        # Group data by basin for colored scatter points
        basin_data = {}
        for data in plot_data:
            for i, basin_name in enumerate(data['basin_names']):
                if basin_name not in basin_data:
                    basin_data[basin_name] = {'x': [], 'y': [], 'years': []}
                basin_data[basin_name]['x'].append(str(data['year']))
                basin_data[basin_name]['y'].append(data['values'][i])
                basin_data[basin_name]['years'].append(data['year'])
        
        # Add scatter trace for each basin
        for basin_name, basin_info in basin_data.items():
            basin_color = colors.get(basin_name, 'gray')
            fig.add_trace(go.Scatter(
                x=basin_info['x'],
                y=basin_info['y'],
                mode='markers',
                name=f'Basin {basin_name}',
                marker=dict(
                    color=basin_color,
                    size=8,
                    line=dict(width=1, color='white'),
                    opacity=0.8
                ),
                hovertemplate=f'<b>Basin {basin_name}</b><br>' +
                             'Year: %{x}<br>' +
                             f'{statistic.title()}: %{{y:.4f}}<br>' +
                             '<extra></extra>',
                legendgroup='basins'
            ))
    else:
        # If no colors provided, add simple scatter points
        for data in plot_data:
            fig.add_trace(go.Scatter(
                x=[str(data['year'])] * len(data['values']),
                y=data['values'],
                mode='markers',
                name='Data Points',
                marker=dict(color='blue', size=6),
                text=data['basin_names'],
                hovertemplate='<b>%{text}</b><br>' +
                             'Year: %{x}<br>' +
                             f'{statistic.title()}: %{{y:.4f}}<br>' +
                             '<extra></extra>',
                showlegend=False
            ))
    
    # Update layout
    fig.update_layout(
        title=f'Distribution of {statistic.title()} Across Basins by Year<br><sub>Individual points colored by basin</sub>',
        xaxis_title='Year',
        yaxis_title=f'{statistic.title()} Value',
        width=1200,
        height=700,
        template='plotly_white',
        legend=dict(
            title="Basins",
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=1.01
        )
    )
    
    return fig

def create_interactive_box_whiskers_plot(residuals, colors=None):
    """
    Create an interactive box and whiskers plot with dropdown for different statistics.
    """
    from ipywidgets import interact, Dropdown, VBox
    
    # Create dropdown for statistics
    statistic_options = [
        ('Residual', 'residual'),
        ('Mean', 'mean'), 
        ('Standard Deviation', 'std'), 
        ('RMS', 'rms'),
        ('Winsorized Mean', 'winsorized_mean'),
        ('Outlier Weighted Mean', 'outlier_weighted_mean'),
        ('Sum', 'sum')
    ]
    
    statistic_dropdown = Dropdown(
        options=statistic_options,
        value='residual',
        description="Statistic:"
    )
    
    def interactive_box_plot(statistic):
        """Interactive box plotting function"""
        fig = create_box_whiskers_plot(residuals, statistic, colors=colors)
        if fig:
            fig.show()
    
    interact(interactive_box_plot, statistic=statistic_dropdown)

# Create the box and whiskers plot
print("Creating box and whiskers plot...")
create_interactive_box_whiskers_plot(residuals, colors=colors)

Creating box and whiskers plot...


interactive(children=(Dropdown(description='Statistic:', options=(('Residual', 'residual'), ('Mean', 'mean'), …